In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import datetime as dt
import utils.helper as helper
import utils.eval as eval

import os
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import importlib
import utils.helper as helper
importlib.reload(helper)

### load  incidence and meta data from converted rds files

In [ ]:
data_path = "data/2_processed/covid_incidence.csv"
incidence = pd.read_csv(data_path, parse_dates=True, index_col=0)

model = "arima"
state = "az"

data_path = f"data/3_incidence predictions/{model}/"

arr = np.load(f"{data_path}{state}_forecasts.npy")
meta = np.load(f"{data_path}{state}_meta.npz")
start_dates_days = meta["start_dates"]
dates_days = meta["dates"]

# convert int days since 1970-01-01 to datetime.date
to_date = lambda d: dt.date(1970, 1, 1) + dt.timedelta(days=int(d))
start_dates = np.array([to_date(d) for d in start_dates_days])
start_dates

In [ ]:
states = ["az","ca","il","md","nj","ny"]
models = ["arima","gp","prophet","deepar","chronos2"]

## Evaluation

### AUC

In [ ]:
def load_concat(horizon_label):
    """Return a tidy df with columns: date, state, model, horizon, y, p."""
    rows = []
    pcol = "peak_prob_1w" if horizon_label == "1w" else "peak_prob_2w"
    for state in states:
        # ground truth
        y_df = (pd.read_csv(gt_dir / f"{state}_test.csv", index_col=0, parse_dates=True)
                  .rename(columns={"peak":"y"}))
        y_df.index.name = "date"

        for model in models:
            pr_path = pred_dir / model / f"{state}_{horizon_label}_forecast.csv"
            if not pr_path.exists():
                continue
            p_df = (pd.read_csv(pr_path, index_col=0, parse_dates=True)
                      .rename(columns={pcol:"p"}))
            p_df.index.name = "date"
            df = (y_df.join(p_df, how="inner")
                        .loc[:, ["y","p"]]
                        .dropna())
            if df.empty: 
                continue
            df = df.assign(state=state, model=model, horizon=horizon_label)
            df = df.reset_index()[["date","state","model","horizon","y","p"]]
            rows.append(df)
    if not rows:
        return pd.DataFrame(columns=["date","state","model","horizon","y","p"])
    return pd.concat(rows, ignore_index=True)

In [ ]:
from pathlib import Path
from matplotlib import gridspec

pred_dir = Path("data/4_peak time predictions")
gt_dir   = Path("data/5_test data")
out_dir  = Path("results"); out_dir.mkdir(parents=True, exist_ok=True)

# ---------- LOAD & STACK ----------
# Return a tidy df with columns: date, state, model, horizon, y, p.
df_1w = load_concat("1w")
df_2w = load_concat("2w")
df_all = pd.concat([df_1w, df_2w], ignore_index=True)
# Make sure types are nice
df_all["y"] = df_all["y"].astype(int)
df_all["p"] = df_all["p"].astype(float)

# ---------- ROC & AUC ----------
rocs = eval.compute_rocs(df_all)

# ---------- AUC TABLE ----------
auc_rows = []
for m in models:
    auc_1 = rocs.get(("1w", m), {}).get("auc", np.nan)
    auc_2 = rocs.get(("2w", m), {}).get("auc", np.nan)
    auc_rows.append({
        "Model": m,
        "1-week AUC": None if np.isnan(auc_1) else int(round(100*auc_1)),
        "2-week AUC": None if np.isnan(auc_2) else int(round(100*auc_2)),
    })
auc_tbl = pd.DataFrame(auc_rows)

# ---------- PLOT ----------
# layout: 1 row, 3 columns → [ROC 1w] [ROC 2w] [AUC table]
fig = plt.figure(figsize=(12, 4.8), dpi=160)
gs = gridspec.GridSpec(nrows=1, ncols=3, width_ratios=[1.2, 1.2, 0.9], wspace=0.35)

def style_axes(ax, title):
    ax.plot([0,1],[0,1], linestyle="--", alpha=0.5)  # diagonal
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    # show ticks as percents (like your R code)
    xt = np.linspace(0,1,6); yt = np.linspace(0,1,6)
    ax.set_xticks(xt); ax.set_xticklabels((xt*100).astype(int))
    ax.set_yticks(yt); ax.set_yticklabels((yt*100).astype(int))
    ax.set_xlabel("1 - Specificity (%)")
    ax.set_ylabel("Sensitivity (%)")
    ax.set_title(title)

# Panel: 1-week
ax1 = fig.add_subplot(gs[0,0])
for m in models:
    item = rocs.get(("1w", m))
    if not item: 
        continue
    ax1.plot(item["fpr"]*100/100.0, item["tpr"]*100/100.0, label=m)  # keep in [0,1] domain
style_axes(ax1, "1-week ahead forecast")
ax1.legend(loc="lower right", frameon=False, fontsize=9)

# Panel: 2-week
ax2 = fig.add_subplot(gs[0,1])
for m in models:
    item = rocs.get(("2w", m))
    if not item:
        continue
    ax2.plot(item["fpr"]*100/100.0, item["tpr"]*100/100.0, label=m)
style_axes(ax2, "2-week ahead forecast")

# Table on the right
ax3 = fig.add_subplot(gs[0,2])
ax3.axis("off")
# Convert to string, show integers or "--"
tbl_disp = auc_tbl.copy()
for col in ["1-week AUC","2-week AUC"]:
    tbl_disp[col] = tbl_disp[col].apply(lambda x: "--" if x is None else f"{x}")
table = ax3.table(cellText=tbl_disp.values,
                  colLabels=tbl_disp.columns,
                  loc="center",
                  cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.2)

fig.tight_layout()
#fig.savefig(out_dir / "peak_rocs.pdf", bbox_inches="tight")
#fig.savefig(out_dir / "peak_rocs.png", bbox_inches="tight")
plt.close(fig)

print("Saved:", out_dir / "peak_rocs.pdf")
print(auc_tbl)

In [ ]:
def plot_peak_forecast_with_observed_peaks(
    state: str,
    horizon: str,              # "1w" or "2w"
    model_name: str,           # just for the legend/title
    pred_dir: str,             # directory containing the new model's forecasts
    peak_col_1w: str = "peak_prob_1w",
    peak_col_2w: str = "peak_prob_2w",
    test_dir: str = "data/5_test data",
):

    pred_path = Path(pred_dir) / f"{state}_{horizon}_forecast.csv"
    test_path = Path(test_dir) / f"{state}_test.csv"

    # --- Load forecast ---
    pred_df = pd.read_csv(pred_path, index_col=0, parse_dates=True)
    if horizon == "1w":
        # unify column name
        pred_df = pred_df.rename(columns={peak_col_1w: "peak_prob"})
    elif horizon == "2w":
        pred_df = pred_df.rename(columns={peak_col_2w: "peak_prob"})
    else:
        raise ValueError("horizon must be '1w' or '2w'")

    # --- Load observed peaks ---
    test_df = pd.read_csv(test_path, index_col=0, parse_dates=True)
    # ensure column name is 'peak'
    if "peak" not in test_df.columns:
        # if your test data uses a different name, adapt here
        raise KeyError("Expected a 'peak' column in the test data.")

    # --- Align by date ---
    merged = pred_df.join(test_df[["peak"]], how="inner")
    merged = merged.sort_index()

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(10, 4), dpi=140)

    # forecast time series
    ax.plot(
        merged.index,
        merged["peak_prob"],
        label=f"{model_name} {horizon} peak forecast",
    )

    ax.set_xlabel("Date")
    ax.set_ylabel("Peak probability")
    ax.set_title(f"{model_name} {horizon} peak-time forecast for {state.upper()}")

    # vertical lines at observed peaks
    peak_dates = merged.index[merged["peak"] == 1]
    first = True
    for d in peak_dates:
        ax.axvline(d, color="grey", linestyle="--", alpha=0.5,
                   label="Observed peak" if first else None)
        first = False

    ax.legend(loc="upper left", frameon=False)
    ax.grid(alpha=0.3)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_peak_forecast_with_observed_peaks(
    state="az",
    horizon="1w",
    model_name="NewModelX",
    pred_dir="data/4_peak time predictions"
)

### Brier score

In [ ]:
# ---------- LOAD & STACK (reuse if you already defined it) ----------
df_1w = load_concat("1w")
df_2w = load_concat("2w")
df_all = pd.concat([df_1w, df_2w], ignore_index=True)

# ---------- METRICS: Brier & Brier Skill Score ----------
bs_df = eval.compute_bs_table(df_all)

# For plotting convenience
bs_1w = (bs_df.query("horizon=='1w'")
         .set_index("model").reindex(models).reset_index())
bs_2w = (bs_df.query("horizon=='2w'")
         .set_index("model").reindex(models).reset_index())

# ---------- TABLE (percent formatting like the ROC table) ----------
tbl = pd.DataFrame({
    "Model": models,
    "1-w BS":  (bs_1w["brier"]).round(4),
    "2-w BS":  (bs_2w["brier"]).round(4),
    "1-w BSS": (bs_1w["bss"]).round(3),
    "2-w BSS": (bs_2w["bss"]).round(3),
})

# ---------- PLOT ----------
# layout: 1 row, 3 columns → [Bars 1w] [Bars 2w] [Table]
fig = plt.figure(figsize=(12, 4.8), dpi=160)
gs = gridspec.GridSpec(nrows=1, ncols=3, width_ratios=[1.2, 1.2, 1.1], wspace=0.35)

def add_bar_panel(ax, df_slice, title):
    vals = (df_slice["brier"]).values  # percent BS
    ax.bar(df_slice["model"], vals)
    ax.set_ylabel("Brier score")
    ax.set_title(title)
    ax.set_ylim(0, max(1.05 * np.nanmax(vals), 1e-6))
    # show values on top of bars
    for i, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9, rotation=0)
    ax.tick_params(axis='x', labelrotation=20)

# Panels
ax1 = fig.add_subplot(gs[0,0]); add_bar_panel(ax1, bs_1w, "1-week ahead forecast")
ax2 = fig.add_subplot(gs[0,1]); add_bar_panel(ax2, bs_2w, "2-week ahead forecast")

# Side table
ax3 = fig.add_subplot(gs[0,2]); ax3.axis("off")
disp = tbl.copy()
table = ax3.table(cellText=disp.values,
                  colLabels=disp.columns,
                  loc="center",
                  cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.2)

fig.tight_layout()
fig.savefig(out_dir / "peak_brier.pdf", bbox_inches="tight")
fig.savefig(out_dir / "peak_brier.png", bbox_inches="tight")
plt.close(fig)

print("Saved:", out_dir / "peak_brier.pdf")
print(tbl)

In [ ]:
state = "ca"
model = "prophet"

onew = pd.read_csv(f"data/4_peak time predictions/{model}/{state}_1w_forecast.csv", index_col=0, parse_dates=True)

# plot onew distribution
plt.figure(figsize=(10,6))
plt.plot(onew.index, onew['peak_prob_1w'], label='1-Week Ahead Peak Probability')
plt.xlabel('Date')
plt.ylabel('Peak Probability')
plt.title(f'1-Week Ahead Peak Probability Forecast for {state.upper()} using {model.upper()} Model')
plt.legend()
plt.grid()
plt.show()

### Elementary score

In [ ]:
def keep_low_draws(draws: np.ndarray, alpha: float) -> np.ndarray:
    W, H, D = draws.shape
    K = max(1, int(np.floor((1.0 - alpha) * D)))  # ensure >=1
    # partition so that the first K entries along axis=2 are the K smallest (order within K arbitrary)
    part = np.partition(draws, K - 1, axis=2)
    return part[..., :K].astype(np.float32, copy=False)

In [ ]:
P_window = {"window_len": 14, "stride_size": 14, "n_windows": 51}
P_peak   = {"peak_window": 11, "rel_thr": 0.05}
P_ma     = {"ma": 7}
qt_levels = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95], dtype=np.float64)
seed = 123
start_indices = helper.get_window_start_indices_from_dates(incidence.index, start_dates)


alpha = 0.1
theta = 0.8

# Define function to compute elementary scores
def compute_elementary_scores(models, states, alpha, theta):
    mean_ele_scores = {}
    for model in models:
        #print(f"Processing model: {model}")
        ele_score = {"one_w": [], "two_w": []}
        for state in states:
            
            y_obs = incidence[[state]].to_numpy()[:, 0]              # (n_rows,)
            obs_win = helper.split_to_supple(y_obs, start_indices, P_peak)  # (W, half+6, 1)

            # Process the model forecasts
            forecast = np.load(f"data/predictions/{model}/{state}_forecasts.npy")
            if model == "chronos2":
                draws = helper.sample_draws_from_quantiles(forecast, qt_levels, n_draws=2000, seed=seed)
            else:
                draws = forecast.astype(np.float32)

            # shrink along the draws axis using the (1 - alpha) upper-quantile rule
            draws_kept = keep_low_draws(draws, alpha)  # (W,14,K)

            # peak-time probabilistic forecast (unchanged, reusing your validated function)
            peak_probs = helper.compute_windowed_inci_peak(draws_kept, obs_win, P_peak)

            one_w, two_w = helper.split_week_forecast(peak_probs, start_indices, incidence.index, window_len=14, stride_size=7)

            test_df = pd.read_csv(f"data/test data/{state}_test.csv", index_col=0, parse_dates=True)

            # compute for one_w
            merged_df = one_w.join(test_df, how="inner")
            merged_df.columns = ['peak_prob', 'actual_peak']
            mean_prob = merged_df['peak_prob'].mean()
            merged_df["forecast_peak"] = (merged_df['peak_prob'] >= (1+theta)*mean_prob).astype(int)
            merged_df["elementary_score"] = alpha * ((merged_df["forecast_peak"] == 1) & (merged_df["actual_peak"] == 0)) + (1-alpha) * ((merged_df["forecast_peak"] == 0) & (merged_df["actual_peak"] == 1))
            mean_ele = merged_df["elementary_score"].mean()
            ele_score["one_w"].append(mean_ele)

            # compute for two_w
            merged_df = two_w.join(test_df, how="inner")
            merged_df.columns = ['peak_prob', 'actual_peak']
            mean_prob = merged_df['peak_prob'].mean()
            merged_df["forecast_peak"] = (merged_df['peak_prob'] >= (1+theta)*mean_prob).astype(int)
            merged_df["elementary_score"] = alpha * ((merged_df["forecast_peak"] == 1) & (merged_df["actual_peak"] == 0)) + (1-alpha) * ((merged_df["forecast_peak"] == 0) & (merged_df["actual_peak"] == 1))
            mean_ele = merged_df["elementary_score"].mean()
            ele_score["two_w"].append(mean_ele)
        
        avg_ele_score = {key: np.mean(values) for key, values in ele_score.items()}
        mean_ele_scores[model] = avg_ele_score
    
    return mean_ele_scores

mean_elementary_scores = compute_elementary_scores(models, states, alpha, theta)

### Murphy diagram

In [ ]:
_es_cache = {}

# -----------------------------------
# 1) SINGLE-RUN elementary score core
# -----------------------------------
def mean_elementary_scores_for(models, states, alpha, theta, use_cache=True):
    """
    Wrapper around your existing compute_elementary_scores().
    No new logic; just optional caching keyed by (alpha, theta).
    """
    key = (round(float(alpha), 8), round(float(theta), 8))
    if use_cache and key in _es_cache:
        return _es_cache[key]
    res = compute_elementary_scores(models, states, alpha, theta)  # <-- YOUR function
    if use_cache:
        _es_cache[key] = res
    return res

# -----------------------------------
# 2) SWEEPS to build Murphy diagrams
# -----------------------------------
def sweep_alpha(models, states, theta_fixed, alphas):
    rows = []
    for a in alphas:
        print(f"Sweeping alpha={a}...")
        res = mean_elementary_scores_for(models, states, alpha=a, theta=theta_fixed)
        for m in models:
            rows.append({
                "param": "alpha", "value": a, "model": m,
                "one_w": res[m]["one_w"], "two_w": res[m]["two_w"]
            })
    return pd.DataFrame(rows)

def sweep_theta(models, states, alpha_fixed, thetas):
    rows = []
    for t in thetas:
        print(f"Sweeping theta={t}...")
        res = mean_elementary_scores_for(models, states, alpha=alpha_fixed, theta=t)
        for m in models:
            rows.append({
                "param": "theta", "value": t, "model": m,
                "one_w": res[m]["one_w"], "two_w": res[m]["two_w"]
            })
    return pd.DataFrame(rows)

# -----------------------------------
# 3) PLOTTING (two panels + consistent style)
# -----------------------------------
def plot_murphy(df, param_label, outpath):
    """df columns: ['param','value','model','one_w','two_w']"""
    fig = plt.figure(figsize=(12, 4.2), dpi=160)
    gs = gridspec.GridSpec(1, 2, width_ratios=[1,1], wspace=0.25)

    def add_panel(ax, ycol, title):
        for m in df["model"].unique():
            sub = df[df["model"] == m].sort_values("value")
            ax.plot(sub["value"], sub[ycol], label=m)
        ax.set_xlabel(param_label)
        ax.set_ylabel("Mean elementary score s(α, θ)")  # lower is better (loss)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[0,0]); add_panel(ax1, "one_w", "1-week ahead forecast")
    ax2 = fig.add_subplot(gs[0,1]); add_panel(ax2, "two_w", "2-week ahead forecast")
    ax1.legend(loc="upper right", frameon=False, fontsize=9)

    fig.tight_layout()
    out = Path(outpath)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out.with_suffix(".pdf"))

In [ ]:
# Event-specific Murphy: fix θ, sweep α
alphas = np.linspace(0.1, 0.9, 20)   # keep <1 so at least one draw remains
theta_fixed = 0.80
df_alpha = sweep_alpha(models, states, theta_fixed, alphas)
plot_murphy(df_alpha, param_label="α", outpath="results/murphy_alpha")

# User-specific Murphy: fix α, sweep θ
thetas = np.linspace(0.3, 2.0, 20)     # adjust to your decision-rule range
alpha_fixed = 0.10
df_theta = sweep_theta(models, states, alpha_fixed, thetas)
plot_murphy(df_theta, param_label="θ", outpath="results/murphy_theta")

### Summary score

In [ ]:
def compute_summary_scores(models, states, alpha_samples, theta_samples, seed=123):
    assert len(alpha_samples) == len(theta_samples), "alpha/theta arrays must have same length"
    np.random.seed(seed)
    N = len(alpha_samples)

    # accumulate elementary scores for each model across all sample pairs
    acc = {m: {"one_w": [], "two_w": []} for m in models}

    for i in range(N):
        a, t = float(alpha_samples[i]), float(theta_samples[i])
        print(f"Sampling {i+1}/{N}: α={a:.3f}, θ={t:.3f}")
        res = compute_elementary_scores(models, states, alpha=a, theta=t)
        for m in models:
            acc[m]["one_w"].append(res[m]["one_w"])
            acc[m]["two_w"].append(res[m]["two_w"])

    # average over all samples
    summary_scores = {
        m: {
            "one_w": float(np.mean(acc[m]["one_w"])) if acc[m]["one_w"] else np.nan,
            "two_w": float(np.mean(acc[m]["two_w"])) if acc[m]["two_w"] else np.nan,
        }
        for m in models
    }
    return summary_scores

In [ ]:
N = 30
alpha_samples = np.random.uniform(0.1, 0.5, size=N)
theta_samples = np.random.normal(0.8, 0.2, size=N)

summary_scores = compute_summary_scores(models, states, alpha_samples, theta_samples)
print(summary_scores)